# 10 · Multilingual reranking

Two off-the-shelf multilingual cross-encoders (no fine-tuning, settings fixed before evaluation) rerank identical frozen candidate pools (≤ 20 chunks) from dense and hybrid first stages. Local inference only (Apple M1: first stage on CPU, reranking on the GPU via MPS; a few early rows ran on CPU and are reported separately); nothing is deployed.

**Inputs** are versioned files under `research/data/` and `research/configs/`; nothing here connects to the production database or writes to it. Run top to bottom (Kernel → Restart & Run All).

In [ ]:
import sys, json, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RESEARCH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "research"
sys.path.insert(0, str(RESEARCH / "scripts"))
import analysis as A

A.style()
np.random.seed(A.SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
EXPERIMENTS = json.loads((RESEARCH / "configs/experiments.json").read_text())
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__} | seed {A.SEED}")
print(f"dataset snapshot: {A.SNAPSHOT} (set RAG_SNAPSHOT=corpus_v1|corpus_v2) | results: {A.RESULTS.relative_to(RESEARCH)} | "
      f"figures: {A.FIGURES.relative_to(RESEARCH)}")

In [ ]:
import glob
def load_exp(ds):
    f = A.RESULTS / f"experiments_{ds}.csv"
    if not f.exists():
        return None
    df = A.load_retrieval(f.name)
    df["latency_ms"] = df["total_ms"]
    return df
EXP = {ds: load_exp(ds) for ds in ("pilot_v0.1", "heldout_v0.2")}
for ds, df in EXP.items():
    print(ds, "not evaluated (no experiments file)" if df is None else f"{len(df)} rows, methods: {sorted(df['method'].unique())}")
MANIFESTS = {Path(f).stem.replace('reranked_', '').replace('.manifest', ''): json.loads(Path(f).read_text())
             for f in glob.glob(str(A.RESULTS / 'reranked_*.manifest.json'))}

In [ ]:
sys.path.insert(0, str(RESEARCH / "scripts"))
from reranking import MODELS
facts = pd.DataFrame([dict(key=k, repository=v["repo"], license=v["license"], parameters=v["params"], base=v["base"],
                           load_s=MANIFESTS.get(k, {}).get("load_seconds"), peak_rss_mb=MANIFESTS.get(k, {}).get("peak_rss_mb"),
                           torch_threads=MANIFESTS.get(k, {}).get("torch_threads"), machine=MANIFESTS.get(k, {}).get("machine"))
                      for k, v in MODELS.items()])
facts

Models considered but not tested: `jinaai/jina-reranker-v2-base-multilingual` (CC-BY-NC-4.0 - non-commercial, not usable for the service) and `Alibaba-NLP/gte-multilingual-reranker-base` (Apache-2.0; needs transformers ≥ 5.18 - a candidate for a follow-up).

In [ ]:
order = ["B0", "B0F", "B0FR_mminilm", "B0FR_bge-m3", "E1", "E2", "E3_mminilm", "E3_bge-m3",
         "E2S", "E4_mminilm", "E4_bge-m3"]
TAB = {}
for ds, df in EXP.items():
    if df is None:
        continue
    s = A.summarize(df, ["method"], metrics=("r1", "r3", "r5", "r10", "rr", "ndcg5", "ndcg10"))
    s = s.set_index("method").reindex([m for m in order if m in set(s["method"])]).reset_index()
    s.insert(0, "dataset", ds); TAB[ds] = s
res = pd.concat(TAB.values(), ignore_index=True)
res.to_csv(A.RESULTS / "m5_retrieval_summary.csv", index=False)
res[["dataset", "method", "n", "r1", "r1_lo", "r1_hi", "r3", "r5", "r10", "rr", "rr_lo", "rr_hi", "ndcg5", "ndcg10"]].round(3)

In [ ]:
COL = {"B0": A.COLORS["B0"], "E2": A.COLORS["E2"], "E3_mminilm": A.COLORS["E1"], "E3_bge-m3": A.COLORS["E2S"]}
show = ["B0", "E2", "E3_mminilm", "E3_bge-m3"]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3), sharey=True)
for ax, ds in zip(axes, ["pilot_v0.1", "heldout_v0.2"]):
    if ds not in TAB:
        ax.text(0.5, 0.5, "Not evaluated", ha="center", transform=ax.transAxes); ax.set_xticks([]); ax.set_yticks([]); continue
    t = TAB[ds].set_index("method")
    rows = []
    for m in [m for m in show if m in t.index]:
        for metric, label in (("r1", "Recall@1"), ("r3", "Recall@3"), ("rr", "MRR"), ("ndcg10", "nDCG@10")):
            rows.append(dict(method=m, metric=label, v=t.loc[m, metric], v_lo=t.loc[m, f"{metric}_lo"], v_hi=t.loc[m, f"{metric}_hi"]))
    tab = pd.DataFrame(rows).set_index(["method", "metric"])
    A.grouped_bars(ax, tab, ["Recall@1", "Recall@3", "MRR", "nDCG@10"], [m for m in show if m in t.index], "v", colors=COL,
                   labels={"B0": "B0 dense", "E2": "E2 hybrid", "E3_mminilm": "E3 hybrid + mMiniLM", "E3_bge-m3": "E3 hybrid + bge-m3"})
    role = "development (used for tuning earlier)" if ds.startswith("pilot") else "held-out (inspected, untuned)"
    ax.set_title(f"{ds} - {role}\n(n = {int(t['n'].iloc[0])} questions, 20 scenarios)", fontsize=10.5)
axes[0].set_ylabel("score (95% CI)")
h_, l_ = axes[0].get_legend_handles_labels(); fig.legend(h_, l_, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.08))
fig.tight_layout(); print(A.save(fig, "10_reranking_comparison"))

## Per language (held-out)

In [ ]:
df = EXP["heldout_v0.2"]
if df is not None:
    pl = A.summarize(df[df.method.isin(["B0", "E2", "E3_mminilm", "E3_bge-m3"])], ["method", "language"], metrics=("r1", "rr"))
    display(pl.pivot(index="language", columns="method", values="rr").loc[A.LANGS].round(3))
    tab = pl.set_index(["method", "language"]).rename(index=A.LANG_NAMES, level=1)
    fig, ax = plt.subplots(figsize=(12, 4))
    A.grouped_bars(ax, tab, [A.LANG_NAMES[l] for l in A.LANGS], [m for m in show if m in set(pl.method)], "rr", colors=COL,
                   labels={"B0": "B0 dense", "E2": "E2 hybrid", "E3_mminilm": "E3 + mMiniLM", "E3_bge-m3": "E3 + bge-m3"})
    ax.set_ylabel("MRR (95% CI)"); ax.set_title("MRR by query language - held-out v0.2 (n = 20 questions per language)")
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1)); fig.tight_layout(); print(A.save(fig, "10_reranking_by_language"))

## Latency and resource cost (local Apple M1)

First-stage retrieval runs on the CPU. Reranking ran on the GPU (MPS) except for the first rows of the development set, which ran on the CPU before the device was switched; the table below separates them. Quality-vs-latency uses GPU rows only.

In [ ]:
if "rerank_device" in df:
    dev = df[df.reranker.notna()].groupby(["method", "rerank_device"])["rerank_ms"].agg(["count", "median"])
    display(Markdown("**Reranking time by device** (ms per question; all datasets)"))
    display(pd.concat([EXP[d][EXP[d].reranker.notna()] for d in EXP if EXP[d] is not None])
            .groupby(["reranker", "rerank_device"])["rerank_ms"].agg(["count", "median"]).round(0))
df = df[df.reranker.isna() | (df.get("rerank_device", "mps") != "cpu")]
lat = df.groupby("method")[["first_stage_ms", "rerank_ms", "total_ms"]].median().reindex([m for m in order if m in set(df.method)])
display(lat.round(1).rename(columns=lambda c: c + " (median)"))
pts = TAB["heldout_v0.2"].set_index("method").join(lat)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for m, r in pts.iterrows():
    if pd.isna(r["total_ms"]): continue
    ax.scatter(r["total_ms"], r["rr"], s=60, color=COL.get(m, A.INK2), zorder=3)
    ax.annotate(m, (r["total_ms"], r["rr"]), textcoords="offset points", xytext=(6, 4), fontsize=8, color=A.INK)
ax.set_xscale("log"); ax.set_xlabel("median retrieval + reranking time per question (ms, log scale; Apple M1, reranking on GPU)")
ax.set_ylabel("MRR (held-out)"); ax.set_title("Quality vs latency (held-out v0.2, n = 100)"); ax.grid(axis="both")
fig.tight_layout(); print(A.save(fig, "10_quality_vs_latency"))

In [ ]:
hm = TAB["heldout_v0.2"].set_index("method"); dm = TAB["pilot_v0.1"].set_index("method")
best = max([m for m in hm.index if m.startswith("E3_")], key=lambda m: dm.loc[m, "rr"]) if any(m.startswith("E3_") for m in hm.index) else None
display(Markdown(f"""**Model chosen on the development set**: `{best}` (highest development MRR); its held-out MRR is
{hm.loc[best, 'rr']:.3f} [{hm.loc[best, 'rr_lo']:.3f}, {hm.loc[best, 'rr_hi']:.3f}] vs E2 {hm.loc['E2', 'rr']:.3f}
[{hm.loc['E2', 'rr_lo']:.3f}, {hm.loc['E2', 'rr_hi']:.3f}]. Paired tests are in notebook 12. Reranking cost:
median {lat.loc[best, 'rerank_ms']:.0f} ms per question on this machine's GPU (MPS) (peak memory
{MANIFESTS.get(best.split('_', 1)[1], {}).get('peak_rss_mb', '?')} MB for the reranking process) - compare with Render's
512 MB instance before any deployment decision.

**Limitations.** Off-the-shelf models; pools capped at 20 chunks (the service's `top_k` limit); local Apple M1
timings (GPU reranking) are not production timings - Render has no GPU, and the CPU rows above show the CPU cost;
AI-drafted, unreviewed labels."""))